CPU 离线部署包准备。一次 CPU 会话、30分钟硬超时；不运行完整推理，不正式提交。历史 V1 保留。

In [ ]:
import os,sys,json,time,subprocess,signal
from pathlib import Path
ROOT=Path('/kaggle/working/cpu_bundle')
assert not ROOT.exists(), 'Do not overwrite an existing bundle'
ROOT.mkdir()
os.environ.update(OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',POLARS_MAX_THREADS='2')
(ROOT/'prepare_bundle.py').write_text('"""One preparation process; a parent enforces the 1800 second session budget."""\nimport hashlib,importlib.metadata as md,json,os,shutil,subprocess,sys,time,traceback\nfrom pathlib import Path\nfrom urllib.parse import urlparse,unquote\nROOT=Path(\'/kaggle/working/cpu_bundle\');START=time.monotonic()\ndef sha(p):return hashlib.file_digest(open(p,\'rb\'),\'sha256\').hexdigest()\ndef record(stage,**kw):\n d={\'stage\':stage,\'utc\':time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),\'elapsed_seconds\':time.monotonic()-START,**kw};(ROOT/(stage+\'.json\')).write_text(json.dumps(d,indent=2)+\'\\n\');print(\'BUNDLE\',json.dumps(d),flush=True)\ndef run(args):\n p=subprocess.run(args,capture_output=True,text=True,timeout=max(1,1800-(time.monotonic()-START)));print(p.stdout[-10000:],p.stderr[-5000:],flush=True);p.check_returncode();return p\ntry:\n assert not (ROOT/\'preparation_started.json\').exists(),\'Duplicate preparation prohibited\'\n record(\'preparation_started\',status=\'STARTED\')\n support=Path(\'/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1\')\n if not support.exists():support=Path(\'/kaggle/input/biohub-tracking-support-pack-50ep-v1\')\n assert (support/\'wheels\').is_dir()\n from packaging.utils import parse_wheel_filename\n from packaging.tags import sys_tags\n tags=set(sys_tags());available={}\n for p in sorted((support/\'wheels\').glob(\'*.whl\')):\n  name,ver,_,wtags=parse_wheel_filename(p.name)\n  if wtags & tags:available.setdefault(str(name),[]).append((str(ver),p))\n assert all(len({v for v,p in entries})==1 for entries in available.values()),\'Ambiguous support wheel versions\'\n constraints=[name+\'==\'+entries[0][0] for name,entries in available.items() if name not in [\'numpy\',\'torch\']]\n constraints+=[\'numpy==\'+md.version(\'numpy\'),\'torch==\'+md.version(\'torch\')]\n (ROOT/\'support_constraints.txt\').write_text(\'\\n\'.join(constraints)+\'\\n\')\n specs=json.loads((ROOT/\'dependency_specs.json\').read_text())\n run([sys.executable,\'-m\',\'pip\',\'install\',\'--disable-pip-version-check\',\'--no-index\',\'--find-links\',str(support/\'wheels\'),\'-c\',str(ROOT/\'support_constraints.txt\'),\'--report\',str(ROOT/\'support_install_report.json\'),*specs])\n wheels=ROOT/\'wheels\';wheels.mkdir(exist_ok=True)\n report=json.loads((ROOT/\'support_install_report.json\').read_text())\n selected=[]\n for row in report[\'install\']:\n  src=Path(unquote(urlparse(row[\'download_info\'][\'url\']).path));assert src.is_file() and support in src.parents\n  shutil.copy2(src,wheels/src.name);selected.append(row[\'metadata\'][\'name\']+\'==\'+row[\'metadata\'][\'version\'])\n # One fixed official-PyPI supplement. Dependencies are fixed as well; no environment-wide upgrade.\n run([sys.executable,\'-m\',\'pip\',\'download\',\'--disable-pip-version-check\',\'--index-url\',\'https://pypi.org/simple\',\'--only-binary=:all:\',\'--no-deps\',\'-d\',str(wheels),\'openvino==2026.4.0\',\'openvino-telemetry==2025.2.0\'])\n selected+=[\'openvino==2026.4.0\',\'openvino-telemetry==2025.2.0\']\n (ROOT/\'requirements.lock\').write_text(\'\\n\'.join(sorted(selected))+\'\\n\')\n run([sys.executable,\'-m\',\'pip\',\'install\',\'--disable-pip-version-check\',\'--no-index\',\'--find-links\',str(wheels),\'-r\',str(ROOT/\'requirements.lock\')])\n # Import-only full-chain dependency check; no hidden training/prediction entrypoint.\n check=run([sys.executable,\'-c\',\'import torch,numpy,zarr,polars,tracksdata,pyscipopt,geff,ilpy,blosc2,openvino; import json; print(json.dumps({k:__import__(k).__version__ for k in ["torch","numpy","zarr","polars","openvino"]}))\'])\n record(\'environment\',versions=json.loads(check.stdout.strip().splitlines()[-1]),installed_specs=selected,network_isolation=\'NOT_VERIFIED_PREPARATION_ONLINE\')\n run([sys.executable,str(ROOT/\'convert_bundle.py\')])\n manifest={p.relative_to(ROOT).as_posix():{\'bytes\':p.stat().st_size,\'sha256\':sha(p)} for p in sorted(ROOT.rglob(\'*\')) if p.is_file() and p.name!=\'bundle_manifest.json\'}\n (ROOT/\'bundle_manifest.json\').write_text(json.dumps({\'files\':manifest,\'status\':\'PREPARED_NOT_OFFLINE_VERIFIED\'},indent=2)+\'\\n\')\n record(\'preparation_complete\',status=\'PREPARED\',files=len(manifest),total_bytes=sum(v[\'bytes\'] for v in manifest.values()))\nexcept BaseException as e:\n record(\'preparation_error\',status=\'STOPPED_ERROR\',error_type=type(e).__name__,error=str(e),traceback=traceback.format_exc());raise\n')
(ROOT/'convert_bundle.py').write_text('"""Bounded real-input diagnostic only. Never execute the production notebook."""\nimport ast, contextlib, hashlib, importlib.util, json, math, os, resource, subprocess, sys, time, traceback\nfrom pathlib import Path\nROOT=Path(\'/kaggle/working/cpu_bundle\'); ROOT.mkdir(exist_ok=True)\nSTART=time.monotonic(); DEADLINE=START+1800\n\ndef receipt(stage, **data):\n    data.update(stage=stage,utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),elapsed_seconds=time.monotonic()-START,peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)\n    p=ROOT/(stage+\'.json\'); tmp=p.with_suffix(\'.tmp\');tmp.write_text(json.dumps(data,indent=2,allow_nan=False)+\'\\n\');tmp.replace(p)\n    print(\'RECEIPT\',json.dumps(data,allow_nan=False),flush=True)\n    return data\n\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for b in iter(lambda:f.read(1048576),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef guard():\n    if time.monotonic()>=DEADLINE:raise TimeoutError(\'45 minute conversion/testing budget exhausted\')\n\ndef select_defs(path,names,namespace):\n    tree=ast.parse(path.read_text());nodes=[n for n in tree.body if isinstance(n,(ast.FunctionDef,ast.ClassDef)) and n.name in names]\n    assert {n.name for n in nodes}==set(names)\n    # Exact function/class AST; excludes all training and full-video entrypoints.\n    exec(compile(ast.Module(body=nodes,type_ignores=[]),str(path),\'exec\'),namespace)\n\ndef main():\n    import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, zarr, openvino as ov\n    from torch.utils.checkpoint import checkpoint as grad_ckpt\n    from collections.abc import Sequence\n    torch.set_grad_enabled(False);torch.set_default_dtype(torch.float32)\n    threads=2;torch.set_num_threads(threads);torch.set_num_interop_threads(1)\n    torch.set_float32_matmul_precision(\'highest\')\n    # Tracing-compatible eager attention path, equally used by PT reference and conversion.\n    torch.backends.mha.set_fastpath_enabled(False)\n    for name in [\'matmul\',\'conv\',\'rnn\']:\n        backend=getattr(torch.backends.mkldnn,name,None)\n        if backend is not None and hasattr(backend,\'fp32_precision\'):backend.fp32_precision=\'ieee\'\n    assert not torch.cuda.is_available(), \'CPU session required\'\n    support=Path(\'/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1\')\n    repo=support/\'repo\'; weights=support/\'weights/unet_transformer/split_0/edge_predictor_best.pth\'\n    expected=json.loads((ROOT/\'source_hashes.json\').read_text())\n    actual={n:sha(repo/n) for n in expected}; assert actual==expected, \'support source checksum mismatch\'\n    t=time.monotonic();wh=sha(weights);assert wh==\'12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771\'\n    ns={\'torch\':torch,\'nn\':nn,\'F\':F,\'np\':np,\'Path\':Path,\'json\':json,\'math\':math,\'Sequence\':Sequence,\'grad_ckpt\':grad_ckpt,\'_grad_ckpt\':grad_ckpt,\'_POS_EMBED_DIM\':8,\'DEFAULT_SCALE\':(1.625,.40625,.40625)}\n    select_defs(repo/\'src/biohub_tracking/models/temporal_unet.py\',[\'_conv_block\',\'_TemporalAttention\',\'TemporalUNet3D\'],ns)\n    select_defs(repo/\'src/biohub_tracking/models/simple_node_transformer.py\',[\'CrossAttentionBlock\',\'SimpleNodeTransformer\'],ns)\n    select_defs(repo/\'scripts/train_unet_transformer.py\',[\'UNetNodeTransformer\',\'extract_pos_features\'],ns)\n    ns[\'_DEFAULT_CONFIG\']={\'unet_out_channels\':32,\'unet_layers\':[32,64,128],\'downsample\':[1,4,4],\'window_size\':2}\n    select_defs(repo/\'scripts/predict_unet_transformer.py\',[\'load_model\',\'_load_frame\',\'pool_kernel_from_um\',\'_detect_cells_pooled\'],ns)\n    select_defs(repo/\'src/biohub_tracking/io.py\',[\'_parse_scale\'],ns)\n    W=2;ds=(1,4,4)\n    comp=next(p for p in [Path(\'/kaggle/input/competitions/biohub-cell-tracking-during-development\'),Path(\'/kaggle/input/biohub-cell-tracking-during-development\')] if p.exists())\n    movies=sorted((comp/\'test\').glob(\'*.zarr\'),key=lambda p:p.stem);assert movies\n    movie=movies[0];g=zarr.open_group(str(movie),mode=\'r\'); arr=g[\'0\'];attrs=dict(g.attrs);scale=ns[\'_parse_scale\'](attrs);q=attrs[\'image_statistics\'][\'quantiles\'];ql=float(q[\'0.001\']);qh=float(q[\'0.999\']);shape=list(arr.shape);target=[-(-s//d) for s,d in zip(shape[1:],ds)];assert shape[0]>=W\n    assert movie.stem==\'44b6_0113de3b\'\n    selected=[0]\n    other=[]\n    for path in movies[1:]:\n        shape2=list(zarr.open_group(str(path),mode=\'r\')[\'0\'].shape)\n        if shape2[1:]!=shape[1:]:\n            other=[{\'video\':path.stem,\'raw_shape\':shape2,\'window\':[0,1]}];break\n    manifest={\'video\':movie.stem,\'raw_shape\':shape,\'frames\':list(range(16)), \'original_last_frame\':shape[0]-1,\'segment_end_is_true_end\':False,\'coordinate_mapping\':\'identity original frame and voxel coordinates\',\'window\':W,\'downsample\':list(ds),\'batch\':1,\'selector_mode\':\'FIXED_DIAGNOSTIC_CONFIG\',\'additional_native_shape_window\':other,\'shape_generalization\':\'SELECTED_NOT_RUN\' if other else \'SHAPE_GENERALIZATION_NOT_COVERED\',\'context\':{\'minimum_track\':6,\'rescue_minimum\':4,\'window\':2,\'smoothing_radius\':2,\'segment_length\':16},\'boundary_effect\':\'truncation can affect future association, division, rescue and smoothing; original last-frame protection remains true video end\'}\n    (ROOT/\'sample_manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n    receipt(\'02_samples_frozen\',**manifest)\n    model,W,ds=ns[\'load_model\'](weights,torch.device(\'cpu\'));model.float();assert W==2 and ds==(1,4,4)\n    receipt(\'01_model_loaded\',weight_sha256=wh,source_sha256=actual,load_hash_build_seconds=time.monotonic()-t,window=W,downsample=ds,actual_batch=1,original_cli_unet_batch_size=4,cli_batch_note=\'original predict_video encodes one window; argument unused\',torch=torch.__version__,openvino=ov.__version__,threads=torch.get_num_threads(),interop=torch.get_num_interop_threads(),cuda_available=False,parameter_dtypes=sorted({str(p.dtype) for p in model.parameters()}))\n    xs=[];t=time.monotonic()\n    for s in selected:\n        x=torch.stack([ns[\'_load_frame\'](arr,i,target,ds) for i in range(s,s+W)])\n        x=((x-ql)/(qh-ql+1e-6)).clamp(0).unsqueeze(0).float().contiguous();assert list(x.shape)==[1,W,*target];xs.append(x)\n    receipt(\'03_real_inputs_loaded\',seconds=time.monotonic()-t,shapes=[list(x.shape) for x in xs],dtypes=[str(x.dtype) for x in xs],finite=[bool(torch.isfinite(x).all()) for x in xs],input_sha256=[hashlib.sha256(x.numpy().tobytes()).hexdigest() for x in xs])\n    class Encoder(nn.Module):\n        def __init__(self,m):super().__init__();self.m=m;self.calls=0\n        def forward(self,x):\n            self.calls+=1\n            feat,det=self.m.encode(x)\n            return (feat,*det)\n    enc=Encoder(model).eval()\n    guard();t=time.monotonic();before=enc.calls\n    converted=ov.convert_model(enc,example_input=xs[0],input=list(xs[0].shape))\n    conversion_seconds=time.monotonic()-t\n    receipt(\'05_converted\',seconds=conversion_seconds,example_input_window=selected[0],tracing_forward_calls=enc.calls-before,outputs=len(converted.outputs))\n    assert len(converted.outputs)==3\n    t=time.monotonic();ov.save_model(converted,ROOT/\'encoder.xml\',compress_to_fp16=False)\n    receipt(\'06_saved\',seconds=time.monotonic()-t,compress_to_fp16=False,xml_bytes=(ROOT/\'encoder.xml\').stat().st_size,bin_bytes=(ROOT/\'encoder.bin\').stat().st_size,xml_sha256=sha(ROOT/\'encoder.xml\'),bin_sha256=sha(ROOT/\'encoder.bin\'))\n    del converted\n    core=ov.Core();t=time.monotonic();reloaded=core.read_model(ROOT/\'encoder.xml\');reload_seconds=time.monotonic()-t\n    const_types=sorted({str(n.get_output_element_type(0)) for n in reloaded.get_ops() if n.get_type_name()==\'Constant\'})\n    assert all(\'float16\' not in x and \'bfloat16\' not in x and x not in [\'f16\',\'bf16\'] for x in const_types)\n    opts={\'INFERENCE_PRECISION_HINT\':\'f32\',\'INFERENCE_NUM_THREADS\':threads,\'NUM_STREAMS\':1,\'PERFORMANCE_HINT\':\'LATENCY\'}\n    t=time.monotonic();cm=core.compile_model(reloaded,\'CPU\',opts);compile_seconds=time.monotonic()-t\n    props={k:str(cm.get_property(k)) for k in opts};assert \'f32\' in props[\'INFERENCE_PRECISION_HINT\'] or \'float32\' in props[\'INFERENCE_PRECISION_HINT\']\n    receipt(\'07_reloaded_compiled\',reload_seconds=reload_seconds,compile_seconds=compile_seconds,device=\'CPU\',requested=opts,actual_properties=props,constant_types=const_types,output_types=[str(x.get_element_type()) for x in reloaded.outputs],offline_reload=\'local IR read/compile; network isolation NOT_VERIFIED\')\n    outputs=cm([xs[0].numpy()])\n    receipt(\'08_reload_inference\',output_shapes=[list(outputs[o].shape) for o in cm.outputs],finite=[bool(np.isfinite(outputs[o]).all()) for o in cm.outputs],note=\'preparation connected session; NOT an offline proof\')\n    runtime=cm.get_runtime_model()\n    execution_precisions={}\n    for op in runtime.get_ops():\n        info=op.get_rt_info()\n        if \'runtimePrecision\' in info:\n            k=str(info[\'runtimePrecision\']);execution_precisions[k]=execution_precisions.get(k,0)+1\n    receipt(\'09_precision\',execution_precisions=execution_precisions)\n\nif __name__==\'__main__\':\n    try:main()\n    except BaseException as e:\n        receipt(\'99_error\',error=str(e),traceback=traceback.format_exc());raise\n')
(ROOT/'source_hashes.json').write_text('{\n  "scripts/predict_unet_transformer.py": "c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9",\n  "scripts/train_unet_transformer.py": "c4f6317736bb3bb1ec8f3f6e9a6d935a463e3f0f1f685481b2d13218d35dc9ea",\n  "src/biohub_tracking/models/temporal_unet.py": "d809c35d42f504161074ddeaaa7aee5b407e5bca7f9b4e1d5f9b2ff345666cac",\n  "src/biohub_tracking/models/simple_node_transformer.py": "b97209edeb03840e80d903e3e2a8c81c520641c8ef343f6ca2904d0f80db064e",\n  "src/biohub_tracking/io.py": "efae135b088cecaab463d889f16c885ef6da3ad27b0747327d8ddc28d866b7bd"\n}\n')
(ROOT/'dependency_specs.json').write_text('[\n  "tracksdata",\n  "zarr>=3.0.10,<4",\n  "pyscipopt",\n  "geff>=1.1.3.1.1",\n  "geff-spec<1.2",\n  "ilpy>=0.5.1",\n  "polars>=1.36",\n  "blosc2",\n  "dask",\n  "imagecodecs",\n  "scikit-image>=0.24",\n  "pyarrow",\n  "rustworkx>=0.17.1",\n  "sqlalchemy>=2",\n  "numcodecs>=0.13,<0.16",\n  "donfig>=0.8",\n  "google-crc32c>=1.5",\n  "bidict>=0.23.1",\n  "psygnal>=0.14",\n  "rich",\n  "networkx>=3.2.1",\n  "pydantic>=2.11",\n  "pydantic-core",\n  "annotated-types",\n  "typing-extensions>=4.13",\n  "typing-inspection",\n  "markdown-it-py",\n  "pygments",\n  "click",\n  "cloudpickle",\n  "fsspec",\n  "partd",\n  "locket",\n  "toolz",\n  "pyyaml",\n  "ndindex",\n  "msgpack",\n  "numexpr",\n  "deprecated",\n  "wrapt",\n  "imageio",\n  "pillow",\n  "tifffile",\n  "lazy-loader",\n  "tqdm"\n]\n')
p=subprocess.Popen([sys.executable,str(ROOT/'prepare_bundle.py')],start_new_session=True)
try:
    rc=p.wait(timeout=1800)
    assert rc==0, f'Preparation failed: {rc}'
except BaseException:
    if p.poll() is None: os.killpg(p.pid,signal.SIGKILL)
    raise
finally:
    print('SMALL_RECEIPTS='+json.dumps({f.name:json.loads(f.read_text()) for f in ROOT.glob('*.json') if f.name not in ['support_install_report.json']}))
